# C4 Notebook 07 — Neural Encoder Training and Seven Ablations

**Purpose:** Train independent C4 encoders and seven model configurations on the same **809 train / 269 validation / 351 test** fixed chronological split. The predicted output is a **linear stage-change regression** in metres. The next-day stage = current stage + predicted change.

This is retrospective PP1 proof-of-concept, not evidence of generalization to independent flood events. The original source nominal report at 09:30 is distinct from the simulated 09:00 forecast origin.

In [1]:
# 0 — Imports, repository discovery and source paths
from pathlib import Path
import sys, json, time
import numpy as np
import pandas as pd
import torch

REPO_OVERRIDE = None  # Optional: Path(r'C:\Users\ASUS\projects\rp_project\J26-DS-314')
SRC_OVERRIDE = Path(r'C:\Users\ASUS\projects\rp_project\J26-DS-314\data\processed\c4\aligned')

start = Path(REPO_OVERRIDE).resolve() if REPO_OVERRIDE else Path.cwd().resolve()
ROOT = next((p for p in (start,*start.parents) if (p/'c4_fusion'/'src').exists() and (p/'shared').exists()),None)
if ROOT is None: raise FileNotFoundError('Set REPO_OVERRIDE to your local J26-DS-314 repository root')
sys.path.insert(0,str(ROOT))
from c4_fusion.src.data.dataset import C4Store, C4Dataset
from c4_fusion.src.train import run_model, set_seed
from c4_fusion.src.models.encoders import C4AblationModel

cfg = json.loads((ROOT/'c4_fusion/configs/c4_training.json').read_text())
ALIGNED = Path(SRC_OVERRIDE).resolve() if SRC_OVERRIDE else ROOT/'data/processed/c4/multimodal'
store=C4Store(ROOT,inputs_npz=ALIGNED/'c4_aligned_inputs_1429_PRIVATE.npz',labels_npz=ALIGNED/'c4_aligned_labels_1429_PRIVATE.npz',index_csv=ALIGNED/'c4_aligned_index_1429_PRIVATE.csv')
OUT=ROOT/'results'/'pp1'/'c4_neural';OUT.mkdir(parents=True,exist_ok=True)
print('device:', 'CUDA '+torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Aligned rows and split counts:',len(store.index),store.index.split.value_counts().to_dict())

device: CUDA NVIDIA GeForce RTX 5070 Laptop GPU
Aligned rows and split counts: 1429 {'TRAIN': 809, 'TEST': 351, 'VALIDATION': 269}


In [3]:
# 1 — Contract and modality feasibility preflight (never trains or downloads)
print('Alignment input shapes:', {k:v.shape for k,v in store.inputs.items() if k.startswith('X_')})
print('Forecast period:',store.index.forecast_date.min(),'to',store.index.forecast_date.max())
print('unique SAR patch references:',store.index.patch_relpath.nunique())
print('SAR missing rows:',int((~store.index.sar_available.astype(bool)).sum()))
print('QA flags:',store.index.catchment_qa_status.value_counts().to_dict())
assert len(store.index)==1429

Alignment input shapes: {'X_weather': (1429, 14, 4), 'X_environment': (1429, 9), 'X_ascat_swi': (1429, 3), 'X_ascat_qflag_percent': (1429, 3)}
Forecast period: 2025-11-17 to 2025-12-25
unique SAR patch references: 264
SAR missing rows: 0
QA flags: {'CANDIDATE_PENDING_VISUAL_QA': 1291, 'REVIEW_OFFICIAL_DRAINAGE_DIFFERENCE_GT20': 78, 'REVIEW_DISTANT_OUTLET': 36, 'REVIEW_SOURCE_COORDINATE_DISCREPANCY': 24}


In [4]:
# 2 — Train-only feature normalization. No validation or test rows enter fit().
train_rows=np.flatnonzero(store.index.split.eq('TRAIN').to_numpy())
# This first fit checks numerical branches; SAR statistics will be computed after patch validation.
store.fit(train_rows,compute_sar_stats=False)
store.save_stats(OUT/'train_only_numeric_scalers.npz')
print('Train-only numeric scalers computed; SAR untouched.')

Train-only numeric scalers computed; SAR untouched.


In [5]:
# 3 — Fast no-image shape/gradient smoke check
from torch.utils.data import DataLoader
from c4_fusion.src.train import run_epoch
set_seed(cfg['seed'])
small=next(iter(DataLoader(C4Dataset(store,'TRAIN',require_sar=False),batch_size=2,shuffle=False)))
for name in ('M2','M3','M6'):
    model=C4AblationModel(name)
    pred=model(small)
    assert pred.shape==(2,1) and torch.isfinite(pred).all()
    pred.mean().backward()
    print(name,'forward/backward PASS',pred.detach().view(-1).tolist())

M2 forward/backward PASS [0.10118833929300308, 0.11113940924406052]
M3 forward/backward PASS [-0.15749697387218475, -0.14835190773010254]
M6 forward/backward PASS [-0.2138417661190033, -0.2048472911119461]


### Image cache requirement

In [6]:
# 4 — Verify physical SAR files and fit training-only SAR dB channel scalers
unique_verified=store.check_sar_cache()   # SHA256 + shapes; does not call GEE
assert unique_verified==264, f'Unexpected number of distinct SAR patches: {unique_verified}'
store.fit(train_rows,compute_sar_stats=True)
store.save_stats(OUT/'train_only_all_scalers.npz')
print('PASS: 264 physical SAR NPZ patches verified; SAR mean/std fit to TRAIN only:',store.sar_stats)

PASS: 264 physical SAR NPZ patches verified; SAR mean/std fit to TRAIN only: (array([ -7.5416403, -14.309984 ], dtype=float32), array([3.2293012, 3.2459488], dtype=float32))


In [7]:
# 5.1 — Select models: M2, M3, M6.
RUN_MODELS=['M2','M3','M6']  # CHANGE to ['M1','M4','M5','M7'] after smoke-tested no-image runs
EPOCHS=cfg['epochs']
results=[]
for name in RUN_MODELS:
    print('Starting',name,flush=True)
    res=run_model(store,name,OUT,epochs=EPOCHS,patience=cfg['patience'],batch_size=cfg['batch_size'],
       lr=cfg['learning_rate'],seed=cfg['seed'],pretrained=cfg['pretrained_efficientnet_weights'],
       freeze_backbone=cfg['freeze_efficientnet_backbone'])
    results.append(res)
    print('Completed',name,':',res,flush=True)
pd.DataFrame(results).to_csv(OUT/'latest_run_metrics.csv',index=False)

Starting M2
M2 epoch 01 train MAE=0.6513 validation MAE=0.2352
M2 epoch 02 train MAE=0.6456 validation MAE=0.2439
M2 epoch 03 train MAE=0.6436 validation MAE=0.2508
M2 epoch 04 train MAE=0.6403 validation MAE=0.2524
M2 epoch 05 train MAE=0.6373 validation MAE=0.2554
Completed M2 : {'model': 'M2', 'validation_mae_m': 0.23516880097663978, 'test_mae_m': 0.22011435100877727, 'test_rmse_m': 0.3992710650245644, 'persistence_mae_m': 0.2172298602696158, 'persistence_rmse_m': 0.40661358117189905, 'epochs_completed': 5, 'test_n': 351, 'train_n': 809, 'validation_n': 269}
Starting M3
M3 epoch 01 train MAE=0.6579 validation MAE=0.2293
M3 epoch 02 train MAE=0.6516 validation MAE=0.2414
M3 epoch 03 train MAE=0.6464 validation MAE=0.2434
M3 epoch 04 train MAE=0.6439 validation MAE=0.2581
M3 epoch 05 train MAE=0.6443 validation MAE=0.2534
Completed M3 : {'model': 'M3', 'validation_mae_m': 0.22933776754207327, 'test_mae_m': 0.21373981713951484, 'test_rmse_m': 0.40094665188586787, 'persistence_mae_m': 0

In [8]:
# 5.2 — Select models: M1, M4, M5, M7.
RUN_MODELS = ['M1', 'M4', 'M5', 'M7']
EPOCHS=cfg['epochs']
results=[]
for name in RUN_MODELS:
    print('Starting',name,flush=True)
    res=run_model(store,name,OUT,epochs=EPOCHS,patience=cfg['patience'],batch_size=cfg['batch_size'],
       lr=cfg['learning_rate'],seed=cfg['seed'],pretrained=cfg['pretrained_efficientnet_weights'],
       freeze_backbone=cfg['freeze_efficientnet_backbone'])
    results.append(res)
    print('Completed',name,':',res,flush=True)
pd.DataFrame(results).to_csv(OUT/'latest_run_metrics.csv',index=False)

Starting M1
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\ASUS/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth


100.0%


M1 epoch 01 train MAE=0.6624 validation MAE=0.2265
M1 epoch 02 train MAE=0.6508 validation MAE=0.2287
M1 epoch 03 train MAE=0.6472 validation MAE=0.2366
M1 epoch 04 train MAE=0.6430 validation MAE=0.2399
M1 epoch 05 train MAE=0.6408 validation MAE=0.2595
Completed M1 : {'model': 'M1', 'validation_mae_m': 0.2265395337653426, 'test_mae_m': 0.2140387159021322, 'test_rmse_m': 0.40364806290395266, 'persistence_mae_m': 0.2172298602696158, 'persistence_rmse_m': 0.40661358117189905, 'epochs_completed': 5, 'test_n': 351, 'train_n': 809, 'validation_n': 269}
Starting M4
M4 epoch 01 train MAE=0.6737 validation MAE=0.2277
M4 epoch 02 train MAE=0.6592 validation MAE=0.2268
M4 epoch 03 train MAE=0.6517 validation MAE=0.2291
M4 epoch 04 train MAE=0.6473 validation MAE=0.2418
M4 epoch 05 train MAE=0.6380 validation MAE=0.2654
M4 epoch 06 train MAE=0.6340 validation MAE=0.2812
Completed M4 : {'model': 'M4', 'validation_mae_m': 0.2268002416010683, 'test_mae_m': 0.21571523666466744, 'test_rmse_m': 0.4046

In [9]:
# 6 — Final summary: only models actually trained and saved are included.
models=C4AblationModel.ACTIVE.keys()
metrics=[]
for name in models:
    p=OUT/f'{name}_test_metrics.json'
    if p.exists():metrics.append(json.loads(p.read_text()))
summary=pd.DataFrame(metrics)
if len(summary):
    display(summary.sort_values('model'))
    summary.to_csv(OUT/'c4_ablation_results_completed_models.csv',index=False)
else:print('No saved trained models yet.')
if set(models).issubset(set(summary.model)):
    x=summary.set_index('model');print('M7 - M6 test MAE (m):',float(x.loc['M7','test_mae_m']-x.loc['M6','test_mae_m']))
else:print('Seven-ablation comparison is incomplete until all seven models have genuine saved results.')

,model,validation_mae_m,test_mae_m,test_rmse_m,persistence_mae_m,persistence_rmse_m,epochs_completed,test_n,train_n,validation_n
0,M1,0.226540,0.214039,0.403648,0.21723,0.406614,5,351,809,269
1,M2,0.235169,0.220114,0.399271,0.21723,0.406614,5,351,809,269
2,M3,0.229338,0.213740,0.400947,0.21723,0.406614,5,351,809,269
3,M4,0.226800,0.215715,0.404630,0.21723,0.406614,6,351,809,269
4,M5,0.225880,0.216129,0.406008,0.21723,0.406614,6,351,809,269
5,M6,0.233114,0.215910,0.400967,0.21723,0.406614,6,351,809,269
6,M7,0.227064,0.215193,0.404004,0.21723,0.406614,5,351,809,269


M7 - M6 test MAE (m): -0.0007170354496379705


## Training interpretation and next step

- The **encoder dimensions** describe neural-network activations **learned during training**: satellite **64 SAR + 16 SWI + 8 SAR freshness → 128**, weather **16**, environment **16**. These are not extra downloaded datasets and are not learned on all 1,429 records before splitting.
- Encoders and fusion head train jointly, with frozen EfficientNet pretrained backbone as the initial small-data protocol. A valid trainable 2→3 input adapter is used for two SAR bands.
- The common current-stage scalar is input to all seven heads, and the target change is a continuous number. Always compute the final predicted stage in metres before calculating MAE/RMSE.
- **Limitations:** the held-out December test interval is not independent flood-event evaluation; only 39 stations have labels; observation timestamp 09:00 is a simulated origin and the source report's publication time is unverified. Large stage differences and shared station characteristics can influence aggregate scores.
- Save `c4_ablation_results_completed_models.csv` and diagnostic figures; do not claim all seven were trained if any did not finish. Notebook 08 should load the saved checkpoint and test prediction files rather than retrieving satellite data or rerunning upstream preprocessing.